[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_10_Structured_Outputs.ipynb)

# 🏗️ Lesson 10 — Structured Outputs & Validation
### Phase 2: Advanced AI Engineering | *Your 10th step toward open-source AI engineering*

---

## 🎯 What You'll Learn Today

You've built a full capstone agent (AutoResearcher). It works — but its outputs are **strings**. In production, strings are fragile. You can't reliably parse them, validate them, or pipe them into other systems.

Today you learn the skill that separates **hobby AI projects** from **production AI systems**: making LLMs return **typed, validated, structured data** — reliably, every time.

| Concept | You'll master |
|---------|---------------|
| Why unstructured output breaks | The core problem with free-form LLM text |
| Pydantic models | Define your data contracts as Python classes |
| Anthropic tool_use for extraction | Native structured output via the API |
| The `instructor` library | The cleanest way to get structured outputs |
| Validation + auto-retry | Handle bad outputs gracefully |
| Real extractor agent | Extract entities from any text, structured |

---

## 💡 Why This Matters for Your Open-Source Goal

Every serious AI project (LangChain, AutoGen, CrewAI, Instructor) uses structured outputs internally. When you publish your open-source agent, structured outputs make it:
- **Reliable** — predictable data shapes, not random strings
- **Composable** — outputs become inputs to other functions without brittle parsing
- **Professional** — what production engineers actually write

This is your first Phase 2 lesson. Think of Phase 2 as hardening everything you've built into something you'd be proud to publish.

---
## 🔧 Setup — Run This First

In [ ]:
# Install everything we need
!pip install anthropic pydantic instructor -q

print("✅ Packages installed")

In [ ]:
# Load your Anthropic API key from Colab Secrets
# Steps: click the 🔑 key icon in the left sidebar → Add secret → Name: ANTHROPIC_API_KEY → Value: your key
from google.colab import userdata
import anthropic
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

print("✅ Anthropic client ready")

---
## 📖 Section 1: The Problem — Why Free-Form LLM Output Breaks Everything

Let's start with a concrete failure. Imagine you ask Claude to extract information from a job posting — you want the job title, company, salary, and required skills as structured data.

In [ ]:
# The naive approach — just ask Claude, hope for the best

JOB_POST = """
Senior ML Engineer at NeuralCore Inc.
We are looking for a seasoned ML engineer to join our core AI team in San Francisco (hybrid).
Compensation: $180,000–$220,000/year + equity.
Requirements: 5+ years Python, PyTorch or TensorFlow, experience deploying models to production,
familiarity with RAG pipelines, and strong communication skills.
Bonus: Kubernetes, distributed training.
"""

response = client.messages.create(
    model="claude-haiku-4-5-20251001",  # Haiku is fast and cheap — great for extraction
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": f"Extract the job title, company, salary range, and required skills from this posting:\n\n{JOB_POST}"
    }]
)

raw_output = response.content[0].text
print("RAW LLM OUTPUT:")
print(raw_output)
print()

# Now try to use this as structured data...
try:
    salary_min = int(raw_output.split("$")[1].split(",")[0])  # Brittle parsing!
    print(f"Parsed salary min: {salary_min}")
except Exception as e:
    print(f"💥 Parsing failed: {e}")
    print("This is the problem. LLM output format changes across calls. You can't rely on it.")

### 💡 The Core Problem

Free-form LLM output is **non-deterministic in format**. Sometimes it uses bullet points, sometimes numbered lists, sometimes paragraphs. Salary might be `$180k-220k`, `180,000-220,000`, or `$180K to $220K`.

Any parser you write will break eventually. The solution is to **force the LLM to speak your schema**, not hope it speaks yours.

There are two industry-standard approaches:
1. **Anthropic native tool_use** — use a tool definition as a JSON schema the LLM must fill
2. **`instructor` library** — a thin wrapper that makes this even cleaner with Pydantic

Let's learn both.

---
## 📖 Section 2: Pydantic — Your Data Contract

Before we use either approach, we need to understand **Pydantic**. Pydantic lets you define data shapes as Python classes with types. These become your *contract* — what you expect the LLM to return.

Think of it like Java's `@Data` classes or record types, but with automatic validation.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

# Define your data contract as a Pydantic model
class JobPosting(BaseModel):
    job_title: str = Field(description="The job title / role name")
    company: str = Field(description="The company name")
    location: str = Field(description="City or location, e.g. 'San Francisco (hybrid)'")
    salary_min: Optional[int] = Field(None, description="Minimum salary in USD per year, as integer")
    salary_max: Optional[int] = Field(None, description="Maximum salary in USD per year, as integer")
    required_skills: List[str] = Field(description="List of required technical skills")
    bonus_skills: List[str] = Field(default=[], description="Nice-to-have skills")

# Pydantic validates on creation
test_job = JobPosting(
    job_title="Senior ML Engineer",
    company="NeuralCore Inc.",
    location="San Francisco (hybrid)",
    salary_min=180000,
    salary_max=220000,
    required_skills=["Python", "PyTorch", "RAG"],
    bonus_skills=["Kubernetes"]
)

print("✅ Valid JobPosting:")
print(test_job.model_dump_json(indent=2))
print()

# Type-safe access — no more string parsing!
print(f"Salary range: ${test_job.salary_min:,} – ${test_job.salary_max:,}")
print(f"Skills required: {', '.join(test_job.required_skills)}")

In [ ]:
# Pydantic catches bad data at the boundary
try:
    bad_job = JobPosting(
        job_title="Engineer",
        company=12345,            # Wrong type — should be str
        location="NYC",
        required_skills="Python"  # Wrong type — should be List[str]
    )
except Exception as e:
    print(f"💡 Pydantic caught the error:\n{e}")

# This is exactly what we want — fail fast at the boundary, not deep inside your logic
print()
print("Key insight: Pydantic = your schema enforcer. LLMs must conform to it.")

---
## 📖 Section 3: Approach 1 — Anthropic Native Structured Output (tool_use)

You already know tool use from Lesson 3! Here we use the same mechanism differently — **not to call an external function, but to force a structured JSON response**.

The trick: define a tool called `extract_job_data` whose parameters are exactly your data schema. Tell Claude to *only* use that tool. Claude is forced to return valid JSON matching your schema — because that's the only way it can respond.

In [ ]:
import json

# Define the extraction tool — this is your schema as a tool definition
EXTRACTION_TOOL = {
    "name": "extract_job_data",
    "description": "Extract structured job posting information from text",
    "input_schema": {
        "type": "object",
        "properties": {
            "job_title": {
                "type": "string",
                "description": "The job title / role name"
            },
            "company": {
                "type": "string",
                "description": "The company name"
            },
            "location": {
                "type": "string",
                "description": "City or work location"
            },
            "salary_min": {
                "type": "integer",
                "description": "Minimum annual salary in USD as integer, or null if not mentioned"
            },
            "salary_max": {
                "type": "integer",
                "description": "Maximum annual salary in USD as integer, or null if not mentioned"
            },
            "required_skills": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of required technical skills"
            },
            "bonus_skills": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Nice-to-have bonus skills"
            }
        },
        "required": ["job_title", "company", "location", "required_skills", "bonus_skills"]
    }
}

# Call the API — force tool use with tool_choice
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=500,
    tools=[EXTRACTION_TOOL],
    tool_choice={"type": "tool", "name": "extract_job_data"},  # Force this specific tool
    messages=[{
        "role": "user",
        "content": f"Extract the job information from this posting:\n\n{JOB_POST}"
    }]
)

# Extract the tool call result
tool_use_block = next(b for b in response.content if b.type == "tool_use")
raw_json = tool_use_block.input

print("Raw JSON from Claude:")
print(json.dumps(raw_json, indent=2))

# Parse into our Pydantic model — now we have type safety!
job = JobPosting(**raw_json)
print()
print("✅ Parsed as JobPosting Pydantic model:")
print(f"  Title: {job.job_title}")
print(f"  Company: {job.company}")
print(f"  Salary: ${job.salary_min:,} – ${job.salary_max:,}")
print(f"  Required: {job.required_skills}")
print(f"  Bonus: {job.bonus_skills}")

### 💡 What Just Happened?

Claude didn't return a string. It returned a **structured JSON object** that conforms exactly to your schema. Why? Because `tool_choice: {type: tool, name: extract_job_data}` tells Claude: *"Your only valid response is to call this tool with these parameters."*

This is the native Anthropic approach. It works with any model and requires no extra libraries.

**The downside**: You have to maintain two representations of your schema — the Pydantic model AND the JSON tool definition. That's duplication. The `instructor` library fixes this.

---
## 📖 Section 4: Approach 2 — The `instructor` Library (The Clean Way)

`instructor` is an open-source library (87k+ GitHub stars) that wraps LLM clients and lets you pass a Pydantic model directly as the `response_model`. It auto-generates the tool schema from your Pydantic class, calls the API, and returns a validated Pydantic instance — all in one line.

This is the approach used in production codebases.

In [ ]:
import instructor

# Patch the Anthropic client with instructor
# This adds a response_model parameter to client.messages.create()
iclient = instructor.from_anthropic(anthropic.Anthropic())

# Extract job posting — notice: no tool definition needed, just pass the Pydantic class
job = iclient.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=500,
    response_model=JobPosting,  # ← This is the magic
    messages=[{
        "role": "user",
        "content": f"Extract the job information from this posting:\n\n{JOB_POST}"
    }]
)

# job is already a JobPosting instance — validated, typed, ready to use
print("✅ Returned directly as JobPosting Pydantic instance:")
print(job.model_dump_json(indent=2))
print()
print(f"Required skills count: {len(job.required_skills)}")
print(f"Has Kubernetes in bonus: {'Kubernetes' in job.bonus_skills}")

# 💡 EXPERIMENT: Add a new field to JobPosting (e.g., is_remote: bool) and re-run.
#                instructor will automatically include it in the extraction.

### Comparison: Native vs. instructor

| | Native tool_use | instructor |
|---|---|---|
| Setup | Define tool JSON schema manually | Just pass Pydantic class |
| Schema maintenance | Two places (Pydantic + JSON) | One place (Pydantic only) |
| Auto-retry on failure | Manual | Built-in (`max_retries`) |
| Validation | Manual Pydantic parse | Automatic |
| Best for | Simple cases, no extra deps | Production code |

Both approaches are valid. `instructor` is preferred for anything non-trivial.

---
## 📖 Section 5: Nested & Complex Schemas

Real-world data is nested. Pydantic + instructor handle this beautifully.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional
from enum import Enum

# Enums constrain values to a fixed set
class Seniority(str, Enum):
    JUNIOR = "junior"
    MID = "mid"
    SENIOR = "senior"
    STAFF = "staff"
    PRINCIPAL = "principal"

class WorkMode(str, Enum):
    REMOTE = "remote"
    HYBRID = "hybrid"
    ONSITE = "onsite"

# Nested model
class SalaryRange(BaseModel):
    min_usd: Optional[int] = Field(None, description="Min salary in USD/year")
    max_usd: Optional[int] = Field(None, description="Max salary in USD/year")
    includes_equity: bool = Field(False, description="Whether equity/stock is mentioned")

class EnrichedJobPosting(BaseModel):
    job_title: str
    company: str
    seniority: Seniority = Field(description="Infer seniority level from the posting")
    work_mode: WorkMode = Field(description="Infer work mode: remote/hybrid/onsite")
    salary: SalaryRange
    required_skills: List[str]
    bonus_skills: List[str] = []
    one_line_summary: str = Field(description="A single sentence summary of the role")

# Extract with the richer schema
enriched = iclient.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=600,
    response_model=EnrichedJobPosting,
    messages=[{
        "role": "user",
        "content": f"Extract and enrich the job information from this posting:\n\n{JOB_POST}"
    }]
)

print("✅ Enriched extraction:")
print(enriched.model_dump_json(indent=2))
print()
print(f"Work mode (enum): {enriched.work_mode}")
print(f"Seniority (enum): {enriched.seniority}")
print(f"Salary min: ${enriched.salary.min_usd:,}")
print(f"Has equity: {enriched.salary.includes_equity}")

# 💡 EXPERIMENT: Add a field `red_flags: List[str]` — ask Claude to infer potential red flags
#                from the posting (e.g., missing salary, no mention of PTO, etc.)

---
## 📖 Section 6: Validation + Auto-Retry on Bad Output

Even with structured prompting, LLMs sometimes return data that fails your business logic — not JSON structure errors, but *semantic* errors. For example: a salary_min that's higher than salary_max, or an empty required_skills list when the posting clearly lists skills.

Pydantic lets you add **custom validators**. `instructor` will automatically retry the LLM call with the validation error as feedback if validation fails.

In [ ]:
from pydantic import field_validator, model_validator

class ValidatedJobPosting(BaseModel):
    job_title: str
    company: str
    salary_min: Optional[int] = None
    salary_max: Optional[int] = None
    required_skills: List[str]

    # Validator: skills list must not be empty
    @field_validator("required_skills")
    @classmethod
    def skills_not_empty(cls, v):
        if len(v) == 0:
            raise ValueError("required_skills cannot be empty — at least one skill must be listed")
        return v

    # Cross-field validator: salary_min must be <= salary_max
    @model_validator(mode="after")
    def salary_range_valid(self):
        if self.salary_min and self.salary_max:
            if self.salary_min > self.salary_max:
                raise ValueError(
                    f"salary_min ({self.salary_min}) cannot exceed salary_max ({self.salary_max})"
                )
        return self

# Demonstrate validation catching a bad case
print("Test 1: Empty skills list (should fail validation)")
try:
    bad = ValidatedJobPosting(
        job_title="Engineer",
        company="Corp",
        required_skills=[]  # Intentionally empty
    )
except Exception as e:
    print(f"  ✅ Validation caught: {e}")

print()
print("Test 2: Inverted salary range (should fail validation)")
try:
    bad = ValidatedJobPosting(
        job_title="Engineer",
        company="Corp",
        salary_min=300000,  # Higher than max — wrong!
        salary_max=180000,
        required_skills=["Python"]
    )
except Exception as e:
    print(f"  ✅ Validation caught: {e}")

In [ ]:
# instructor auto-retries when validation fails
# max_retries=3 means: if Claude returns invalid data, show it the error and ask again (up to 3 times)

validated = iclient.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=500,
    response_model=ValidatedJobPosting,
    max_retries=3,  # Auto-retry on validation failure
    messages=[{
        "role": "user",
        "content": f"Extract job info from this posting:\n\n{JOB_POST}"
    }]
)

print("✅ Validated extraction passed all validators:")
print(validated.model_dump_json(indent=2))

# 💡 EXPERIMENT: Add a validator that checks job_title is title-cased.
#                See if instructor corrects lowercase titles automatically.

---
## 📖 Section 7: Real-World Agent — Batch Job Extractor

Now let's put it all together: a small agent that processes multiple job postings, extracts structured data from each, and produces a comparison table. This is a real use case — think job board aggregator, recruiter tool, or salary benchmarker.

In [ ]:
from typing import List
import time

# Multiple job postings to process
JOB_POSTINGS = [
    """
    AI Research Engineer at Anthropic
    Remote-first. $200k–$280k + significant equity.
    Requirements: PhD or equivalent experience, Python, ML frameworks (JAX/PyTorch),
    publications in top venues (NeurIPS/ICML/ICLR), experience with large-scale training.
    Nice to have: RLHF, constitutional AI, interpretability research.
    """,
    """
    Junior ML Engineer at StartupAI
    NYC office, 3 days/week. $95,000–$120,000.
    You'll build ML pipelines, work with LLM APIs (OpenAI, Anthropic), and ship features fast.
    Must have: Python, SQL, REST APIs, basic ML knowledge.
    Bonus: LangChain, RAG, vector databases.
    """,
    """
    Staff Data Scientist at TechCorp
    Fully remote. Compensation: competitive (not disclosed).
    Lead a team of 4 data scientists. Drive ML strategy across product lines.
    Required: 8+ years experience, Python, statistics, A/B testing, stakeholder management.
    Nice to have: LLM experience, causal inference.
    """
]

def extract_job(posting: str) -> EnrichedJobPosting:
    """Extract structured data from a single job posting."""
    return iclient.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=600,
        response_model=EnrichedJobPosting,
        max_retries=2,
        messages=[{
            "role": "user",
            "content": f"Extract and enrich job info from:\n\n{posting}"
        }]
    )

# Process all postings
print("🔄 Extracting structured data from 3 job postings...\n")
jobs = []
for i, posting in enumerate(JOB_POSTINGS):
    job = extract_job(posting)
    jobs.append(job)
    print(f"  ✅ [{i+1}/3] {job.job_title} @ {job.company}")
    time.sleep(0.5)  # Brief pause to be API-rate-limit-friendly

print()
print("=" * 70)
print("📊 COMPARISON TABLE")
print("=" * 70)
print(f"{'Role':<35} {'Seniority':<10} {'Min Salary':<14} {'Mode':<10}")
print("-" * 70)
for job in jobs:
    salary = f"${job.salary.min_usd:,}" if job.salary.min_usd else "undisclosed"
    print(f"{job.job_title[:34]:<35} {job.seniority.value:<10} {salary:<14} {job.work_mode.value:<10}")

print()
print("📝 One-line summaries:")
for job in jobs:
    print(f"  • {job.job_title}: {job.one_line_summary}")

---
## 📖 Section 8: Streaming + Partial Objects (Advanced)

For large extractions that take a few seconds, you can **stream the structured output** — getting a partially-filled Pydantic object that completes field by field. This is great for UI responsiveness.

In [ ]:
# Streaming structured output with instructor
# As fields complete, you get partial objects you can start using immediately

print("🌊 Streaming structured extraction (fields fill in as Claude generates them):\n")

with iclient.messages.stream(
    model="claude-haiku-4-5-20251001",
    max_tokens=600,
    response_model=instructor.Partial[EnrichedJobPosting],  # Partial = stream-friendly
    messages=[{
        "role": "user",
        "content": f"Extract job info from:\n\n{JOB_POSTINGS[0]}"
    }]
) as stream:
    last_fields = set()
    for partial in stream:
        # Check which fields have been filled
        filled = {k for k, v in partial.model_dump().items() if v is not None}
        new_fields = filled - last_fields
        if new_fields:
            for field in new_fields:
                val = getattr(partial, field)
                print(f"  ✏️  {field}: {val}")
        last_fields = filled

print()
print("✅ Streaming complete — all fields extracted")

# 💡 EXPERIMENT: Try streaming with EnrichedJobPosting on your own job description.
#                Watch which fields Claude fills first — usually the obvious ones.

---
## 📖 Section 9: Structured Outputs in Your Agent (Upgrading the Capstone)

Let's think about how this changes your **AutoResearcher Agent** from Lesson 9. Currently, the agent returns research as a string. Here's the pattern for upgrading it:

In [ ]:
# How to upgrade any agent's output to be structured
# This is the pattern — adapt it to your own agents

from pydantic import BaseModel, Field
from typing import List

class ResearchFinding(BaseModel):
    claim: str = Field(description="A specific factual claim from research")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence 0.0–1.0")
    source_hint: str = Field(description="Where this came from (tool name or memory)")

class ResearchReport(BaseModel):
    topic: str = Field(description="The research topic")
    executive_summary: str = Field(description="2-3 sentence high-level summary")
    key_findings: List[ResearchFinding] = Field(description="Individual findings, each verifiable")
    open_questions: List[str] = Field(description="What remains unanswered or uncertain")
    recommended_next_steps: List[str] = Field(description="What to research next")
    confidence_overall: float = Field(ge=0.0, le=1.0, description="Overall report confidence")

# Demo: synthesize a research report from a topic
TOPIC = "The impact of transformer architecture on modern NLP"

report = iclient.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1000,
    response_model=ResearchReport,
    max_retries=2,
    messages=[
        {
            "role": "user",
            "content": f"""You are a research synthesis agent. Based on your knowledge,
produce a structured research report on the following topic: {TOPIC}"""
        }
    ]
)

print(f"📋 RESEARCH REPORT: {report.topic}")
print(f"Overall confidence: {report.confidence_overall:.0%}")
print()
print(f"Executive Summary:\n{report.executive_summary}")
print()
print(f"Key Findings ({len(report.key_findings)}):")
for f in report.key_findings:
    print(f"  [{f.confidence:.0%}] {f.claim}")
print()
print(f"Open Questions:")
for q in report.open_questions:
    print(f"  ❓ {q}")
print()
print(f"Next Steps:")
for s in report.recommended_next_steps:
    print(f"  → {s}")

# 💡 EXPERIMENT: Add a field `is_controversial: bool` and see how Claude classifies the topic.

---
## 🎯 Lesson Summary — What You've Mastered

| Pattern | You can now... |
|---------|----------------|
| Pydantic models | Define typed data contracts as Python classes |
| Native tool_use extraction | Force JSON output via tool definitions |
| `instructor` library | Get Pydantic instances directly from API calls |
| Nested schemas | Model complex, hierarchical data structures |
| Custom validators | Enforce business rules on LLM outputs |
| Auto-retry | Recover from validation failures automatically |
| Streaming partial | Get structured data as it generates |
| Agent output upgrade | Replace string outputs with typed report models |

---

## 🚀 Your Challenge for This Week

**Upgrade your AutoResearcher capstone** (Lesson 9) to use structured outputs:

1. Replace the final string output with a `ResearchReport` Pydantic model
2. Add a `confidence` score to each finding (from the web_search tool result)
3. Add custom validation: if `confidence_overall < 0.5`, add a warning flag `low_confidence: bool = True`
4. Print a formatted structured report at the end

When you've done that, your AutoResearcher returns **machine-readable, validated, typed** results — the hallmark of a production AI system.

---

## 📚 What's Next — Phase 2 Roadmap

You've completed all 9 core lessons. Phase 2 takes you deeper:

| Lesson | Topic | Why It Matters |
|--------|-------|----------------|
| **10 (today)** | Structured Outputs & Validation | Production-quality agent outputs ✅ |
| **11** | LangGraph — Stateful Agent Graphs | Industry-standard agent framework |
| **12** | Fine-tuning Fundamentals | When to go beyond prompting |
| **13** | Multimodal AI — Vision + Text | Understand images, not just text |
| **14** | MCP — Model Context Protocol | Build tools that plug into any AI system |
| **15** | Open-Source Project Publishing | Ship your GitHub portfolio project |

---

## 🔑 Key Libraries to Bookmark

- **[instructor](https://python.useinstructor.com/)** — Structured outputs from LLMs
- **[pydantic](https://docs.pydantic.dev/)** — Data validation and settings management
- **[Anthropic docs — Tool use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)** — Native structured output reference

---
*Lesson 10 of Phase 2 — AI Engineering Curriculum | Gourav Khanijoe | 2026-05-09*